# Persian Voice Phone Directory (MVP) - Colab Notebook
Developer: `Moein Parvizi`

This notebook is an MVP of a **Persian Voice Phone Directory** with the following workflow:

1) **Live audio capture** (browser microphone in Colab) -> `caller.wav` at 16kHz mono  
2) **Persian ASR** using `faster-whisper` with the **`large-v3`** model  
3) **Contact retrieval from CSV** (`Name`, `Role`, `Number`) using rule-based parsing + RapidFuzz scoring (name/role matching, ambiguity handling)  
4) **Offline Persian TTS** using **Piper** with the `fa_IR-ganji_adabi-medium` voice from `rhasspy/piper-voices` (ONNX)  
5) Output is played directly inside Colab.


In [ ]:
# Install core Python packaging tools
!pip -q install -U pip setuptools wheel

# Install ASR dependencies (Whisper + fuzzy matching + audio I/O)
!pip -q install faster-whisper rapidfuzz soundfile pydub

# Install offline TTS dependencies (Piper + Hugging Face download tools)
!pip -q install piper-tts huggingface_hub torchaudio

# Install local microphone recording support (for running outside Colab)
!pip -q install sounddevice

# Install ffmpeg for audio format conversion (webm -> wav)
!apt-get -qq update
!apt-get -qq install -y ffmpeg

# Check GPU availability and active device name
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

## 1) Live Audio Recording (Colab Microphone)

This cell records audio using the browser microphone and generates a standard **16kHz mono WAV** file (compatible with Whisper).

- After you click **Stop**, the file `caller.wav` will be created.  
- If the browser blocks access: go to **Site settings → Microphone → Allow**, then refresh the page.

In [ ]:
import base64
import os
import subprocess
import uuid

from IPython.display import Audio, Javascript, display

# Use Colab browser microphone when available.
try:
    from google.colab import output as colab_output
except Exception:
    colab_output = None

# Use local microphone when running outside Colab.
try:
    import sounddevice as sd
    import soundfile as sf
except Exception:
    sd = None
    sf = None

RECORD_JS = r'''
async function recordAudio() {
  const sleep = (time) => new Promise((resolve) => setTimeout(resolve, time));
  const blobToDataUrl = (blob) =>
    new Promise((resolve) => {
      const reader = new FileReader();
      reader.onloadend = () => resolve(reader.result);
      reader.readAsDataURL(blob);
    });

  const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
  const recorder = new MediaRecorder(stream);
  const chunks = [];

  recorder.ondataavailable = (event) => chunks.push(event.data);
  recorder.start();

  const panel = document.createElement('div');
  panel.style.border = '1px solid #ddd';
  panel.style.padding = '12px';
  panel.style.borderRadius = '10px';
  panel.style.margin = '10px 0';
  panel.innerHTML = '<b>Recording...</b> <span id="timer">0.0</span>s ';

  const stopBtn = document.createElement('button');
  stopBtn.textContent = 'Stop';
  stopBtn.style.marginLeft = '12px';
  stopBtn.style.padding = '6px 12px';
  stopBtn.style.borderRadius = '8px';
  stopBtn.style.border = '1px solid #ccc';
  stopBtn.style.cursor = 'pointer';

  panel.appendChild(stopBtn);
  document.body.appendChild(panel);

  const startedAt = performance.now();
  let running = true;
  (async () => {
    while (running) {
      await sleep(100);
      const dt = (performance.now() - startedAt) / 1000.0;
      const timer = document.getElementById('timer');
      if (timer) timer.textContent = dt.toFixed(1);
    }
  })();

  await new Promise((resolve) => (stopBtn.onclick = resolve));
  running = false;
  recorder.stop();

  await new Promise((resolve) => (recorder.onstop = resolve));
  stream.getTracks().forEach((track) => track.stop());
  panel.remove();

  const blob = new Blob(chunks, { type: 'audio/webm;codecs=opus' });
  return await blobToDataUrl(blob);
}
'''


def record_to_wav(wav_path='caller.wav', duration_sec: int = 6):
    """
    In Colab: record with browser mic (JavaScript MediaRecorder).
    Outside Colab: record with local mic (sounddevice).
    """
    if colab_output is not None:
        display(Javascript(RECORD_JS))
        data_url = colab_output.eval_js('recordAudio()')
        _, b64data = data_url.split(',', 1)

        webm_path = f'/content/{uuid.uuid4().hex}.webm'
        with open(webm_path, 'wb') as f:
            f.write(base64.b64decode(b64data))

        subprocess.run(
            ['ffmpeg', '-y', '-i', webm_path, '-ac', '1', '-ar', '16000', wav_path],
            check=True,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )
        os.remove(webm_path)
        return wav_path

    if sd is None or sf is None:
        raise RuntimeError(
            "sounddevice/soundfile is not available. Run: pip install sounddevice soundfile"
        )

    sample_rate = 16000
    channels = 1
    print(f'\n[Mic] Recording for {duration_sec} seconds... Speak now.')
    audio = sd.rec(
        int(duration_sec * sample_rate),
        samplerate=sample_rate,
        channels=channels,
        dtype='int16',
    )
    sd.wait()

    sf.write(wav_path, audio, sample_rate)
    print('[Mic] Saved recording to:', wav_path)
    return wav_path


# Quick recording test (optional)
# wav_path = record_to_wav('caller.wav')
# print('Saved:', wav_path)
# display(Audio(wav_path))


## 2) Persian ASR with faster-whisper (Whisper large-v3)

**Note:** The ASR model variable is named **`asr_model`** to avoid confusion with TTS models.


In [ ]:
from faster_whisper import WhisperModel

# Load the ASR model with GPU/CPU settings based on runtime availability.
def get_asr_model(model_name="large-v3"):
    try:
        import torch
        has_cuda = torch.cuda.is_available()
    except Exception:
        has_cuda = False

    if has_cuda:
        device = "cuda"
        compute_type = "float16"
    else:
        device = "cpu"
        compute_type = "int8"

    print(f"Loading ASR: {model_name} | device={device} | compute_type={compute_type}")
    return WhisperModel(model_name, device=device, compute_type=compute_type)

# Initialize Persian ASR model once and reuse it for all requests.
asr_model = get_asr_model("large-v3")

try:
    top_roles = "، ".join(sorted(df_contacts["Role"].dropna().unique())[:20])
except Exception:
    top_roles = ""

# Transcribe a WAV file to Persian text.
def transcribe_fa(wav_path: str) -> str:
    segments, info = asr_model.transcribe(
        wav_path,
        language="fa",
        task="transcribe",
        beam_size=8,
        vad_filter=True,
        temperature=0.0,
        condition_on_previous_text=False,
        initial_prompt = (
    "دفترچه تلفن فارسی. کاربر نام یا عنوان شغلی/سمت می‌گوید و شماره می‌خواهد. "
    f"سمت‌های رایج: {top_roles}"
)
    )
    print("Detected:", info.language, getattr(info, "language_probability", None))
    return " ".join(seg.text for seg in segments).strip()


## 3) CSV Contact Retrieval + Matching


In [ ]:
import os
import re

import pandas as pd
from rapidfuzz import fuzz

DIGIT_NORMALIZER = str.maketrans('۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩', '01234567890123456789')
DIGIT_WORDS_FA = {
    '0': 'صِفر',
    '1': 'یک',
    '2': 'دو',
    '3': 'سه',
    '4': 'چهار',
    '5': 'پنج',
    '6': 'شِش',
    '7': 'هفت',
    '8': 'هشت',
    '9': 'نُه',
}


def normalize_fa(s: str) -> str:
    if not s:
        return ''

    s = s.strip().replace('ي', 'ی').replace('ك', 'ک')
    s = re.sub(r'[ًٌٍَُِّْـ]', '', s)
    s = s.replace('\u200c', ' ')
    s = re.sub(r'[^\w\sآ-ی۰-۹]', ' ', s)
    s = re.sub(
        r'\b(|میخواستم|آقای|خانم|جناب|آقا|سرکار|دکتر|مهندس|سلام|وقت بخیر|شماره|را|رو|می خواستم|می خوام|میخوام)\b',
        '',
        s,
    )
    return re.sub(r'\s+', ' ', s).strip()


def number_to_fa_spoken(raw_number: str) -> str:
    digits = str(raw_number).translate(DIGIT_NORMALIZER)
    spoken = [DIGIT_WORDS_FA[ch] for ch in digits if ch.isdigit()]
    return '، '.join(spoken) if spoken else str(raw_number)


def ensure_contacts_csv(path: str = 'contacts.csv') -> str:
    if os.path.exists(path):
        return path

    try:
        from google.colab import files

        print(f'فایل {path} پیدا نشد. لطفاً فایل CSV را آپلود کنید.')
        uploaded = files.upload()
        if uploaded:
            return path if os.path.exists(path) else next(iter(uploaded.keys()))
    except Exception as exc:
        raise FileNotFoundError(f'CSV file not found: {path}') from exc

    raise FileNotFoundError('No CSV uploaded.')


def load_contacts_dataframe(path: str = 'contacts.csv') -> pd.DataFrame:
    csv_path = ensure_contacts_csv(path)
    df = pd.read_csv(csv_path, dtype=str, encoding='utf-8-sig').fillna('')

    required = {'Name', 'Role', 'Number'}
    if not required.issubset(set(df.columns)):
        raise ValueError('CSV must contain Name, Role, Number columns.')

    df = df[['Name', 'Role', 'Number']].copy()
    df['Name'] = df['Name'].astype(str).str.strip()
    df['Role'] = df['Role'].astype(str).str.strip()
    df['Number'] = df['Number'].astype(str).str.strip()

    df = df[(df['Name'] != '') & (df['Number'] != '')].copy()
    if df.empty:
        raise ValueError('No valid contacts loaded from CSV.')

    df['Role'] = df['Role'].replace('', 'دفتر')
    df['name_norm'] = df['Name'].map(normalize_fa)
    df['role_norm'] = df['Role'].map(normalize_fa)
    df['number_spoken'] = df['Number'].map(number_to_fa_spoken)

    # Ensure unique labels per name for downstream selection.
    labels = []
    seen = {}
    for _, row in df.iterrows():
        key = (row['Name'], row['Role'])
        seen[key] = seen.get(key, 0) + 1
        labels.append(row['Role'] if seen[key] == 1 else f"{row['Role']} {seen[key]}")

    df['label'] = labels
    return df


def build_contacts_dict(df: pd.DataFrame):
    contacts = {}
    for _, row in df.iterrows():
        contacts.setdefault(row['Name'], {})[row['label']] = row['number_spoken']
    return contacts


df_contacts = load_contacts_dataframe('contacts.csv')
CONTACTS = build_contacts_dict(df_contacts)
print(f"Loaded {df_contacts['Name'].nunique()} names and {len(df_contacts)} numbers from contacts.csv")

PHONE_TYPE_KEYWORDS = {
    'موبایل': ['موبایل', 'همراه', 'شماره همراه', 'شماره موبایل'],
    'دفتر': ['دفتر', 'محل کار', 'شرکت', 'اداره'],
    'منزل': ['منزل', 'خانه', 'خونه'],
}
REPEAT_KWS = ['دوباره بگو', 'یه بار دیگه', 'یک بار دیگه', 'تکرار کن', 'باز بگو', 'مجدد بگو']


def extract_phone_type(txt_norm: str):
    for ptype, kws in PHONE_TYPE_KEYWORDS.items():
        if any(kw in txt_norm for kw in kws):
            return ptype
    return None


def is_repeat_intent(txt_norm: str) -> bool:
    return any(k in txt_norm for k in REPEAT_KWS)


def _pick_best_with_margin(scored, threshold=70, margin=5):
    if not scored:
        return None, 0, []

    scored = sorted(scored, key=lambda x: x[1], reverse=True)
    top_key, top_score = scored[0]
    if top_score < threshold:
        return None, int(top_score), []

    cands = [k for k, s in scored if s >= top_score - margin and s >= threshold]
    return (top_key if len(cands) == 1 else None), int(top_score), cands


def score_request(df: pd.DataFrame, txt_norm: str):
    # Name scoring on normalized ASR text.
    name_rows = df[['Name', 'name_norm']].drop_duplicates().to_dict('records')
    name_scored = [(r['Name'], fuzz.partial_ratio(r['name_norm'], txt_norm)) for r in name_rows]
    name, name_score, candidate_names = _pick_best_with_margin(name_scored, threshold=70, margin=5)

    # Role scoring on normalized ASR text.
    role_rows = df[['Role', 'role_norm']].drop_duplicates().to_dict('records')
    role_scored = [(r['Role'], fuzz.token_set_ratio(r['role_norm'], txt_norm)) for r in role_rows]
    role_scored.sort(key=lambda x: x[1], reverse=True)

    def role_accepted(score):
        return score >= 85 or (score >= 75 and score >= name_score + 10)

    raw_role_score = int(role_scored[0][1]) if role_scored else 0
    accepted_roles = [(r, int(s)) for r, s in role_scored if role_accepted(s)]

    if accepted_roles:
        role_top = accepted_roles[0][1]
        candidate_roles = [r for r, s in accepted_roles if s >= role_top - 5]
        role = candidate_roles[0] if len(candidate_roles) == 1 else None
        role_score = int(role_top)
    else:
        role, role_score, candidate_roles = None, 0, []

    if name_score >= 85 and name_score >= role_score + 8:
        dominant_type = 'name'
    elif role_score >= 88 and role_score >= name_score + 8:
        dominant_type = 'role'
    elif name_score >= 80 and role_score >= 80:
        dominant_type = 'mixed'
    else:
        dominant_type = 'unknown'

    # Score records based on the dominant signal.
    name_set = set(candidate_names)
    role_set = set(candidate_roles)
    records = []
    for _, row in df.iterrows():
        ns = int(fuzz.partial_ratio(row['name_norm'], txt_norm)) if row['Name'] in name_set else 0
        rs = int(fuzz.token_set_ratio(row['role_norm'], txt_norm)) if row['Role'] in role_set else 0

        if dominant_type == 'name':
            combined = ns
        elif dominant_type == 'role':
            combined = rs
        elif dominant_type == 'mixed':
            combined = int(0.6 * ns + 0.4 * rs)
        else:
            combined = max(ns, rs)

        records.append(
            {
                'name': row['Name'],
                'role': row['Role'],
                'label': row['label'],
                'number_spoken': row['number_spoken'],
                'name_score': ns,
                'role_score': rs,
                'combined_score': int(combined),
            }
        )

    records.sort(key=lambda x: x['combined_score'], reverse=True)
    top_combined = records[0]['combined_score'] if records else 0
    keep_threshold = max(70, top_combined - 5)
    top_records = [r for r in records if r['combined_score'] >= keep_threshold][:5]

    return {
        'name': name,
        'name_score': int(name_score),
        'role': role,
        'role_score': int(role_score),
        'raw_role_score': int(raw_role_score),
        'dominant_type': dominant_type,
        'top_records': top_records,
        'candidate_names': candidate_names,
        'candidate_roles': candidate_roles,
        'top3_name_scores': sorted(name_scored, key=lambda x: x[1], reverse=True)[:3],
        'top3_role_scores': role_scored[:3],
    }


def parse_request(asr_text: str):
    txt_norm = normalize_fa(asr_text)
    repeat = is_repeat_intent(txt_norm)
    ptype = extract_phone_type(txt_norm)

    scored = score_request(df_contacts, txt_norm)
    candidate_records = scored['top_records']
    name_raw = scored['name']
    candidate_names = scored['candidate_names']
    candidate_roles = scored['candidate_roles']
    role_raw = scored['role']
    dominant_type = scored['dominant_type']

    # If unresolved, lock a single name inferred from top records.
    unique_names = []
    for record in candidate_records:
        if record['name'] not in unique_names:
            unique_names.append(record['name'])
    if not name_raw and len(unique_names) == 1:
        name_raw = unique_names[0]

    preferred_label = None
    if len(candidate_records) == 1:
        preferred_label = candidate_records[0]['label']
    elif name_raw:
        same_name = [record for record in candidate_records if record['name'] == name_raw]
        if len(same_name) == 1:
            preferred_label = same_name[0]['label']

    # Keep debug logs at the end for tuning.
    print('ASR(raw): ', asr_text)
    print('ASR(norm):', txt_norm)
    print('Repeat?:  ', repeat)
    print('PhoneType:', ptype)
    print(f"NameScore: {scored['name_score']} | RoleScore(raw={scored['raw_role_score']}, accepted={scored['role_score']})")
    print('Top3NameScores:', scored['top3_name_scores'])
    print('Top3RoleScores:', scored['top3_role_scores'])
    print('Name:', name_raw if name_raw else f"AMBIGUOUS {candidate_names}" if candidate_names else 'NOT FOUND')
    print('Role:', role_raw if role_raw else f"AMBIGUOUS {candidate_roles}" if candidate_roles else 'NOT FOUND')
    if candidate_records:
        top = candidate_records[0]
        print('TopRecord:', f"{top['name']} | {top['role']} | combined={top['combined_score']}")
    print('DominantType:', dominant_type)

    return {
        'phone_type': ptype,
        'name_raw': name_raw,
        'candidate_names': candidate_names,
        'candidate_records': candidate_records,
        'preferred_label': preferred_label,
        'role_dominant_intent': dominant_type == 'role',
    }


## 4) Offline Persian TTS with Piper (fa_IR-ganji_adabi-medium)


In [ ]:
from huggingface_hub import hf_hub_download
import os

VOICE_REPO = "rhasspy/piper-voices"
VOICE_SUBPATH = "fa/fa_IR/ganji_adabi/medium"

os.makedirs("voices", exist_ok=True)

onnx_path = hf_hub_download(
    repo_id=VOICE_REPO,
    filename=f"{VOICE_SUBPATH}/fa_IR-ganji_adabi-medium.onnx",
    local_dir="voices",
    local_dir_use_symlinks=False
)
json_path = hf_hub_download(
    repo_id=VOICE_REPO,
    filename=f"{VOICE_SUBPATH}/fa_IR-ganji_adabi-medium.onnx.json",
    local_dir="voices",
    local_dir_use_symlinks=False
)

print("ONNX:", onnx_path)
print("JSON:", json_path)


In [ ]:
import wave
import torch
from IPython.display import Audio, display
from piper.voice import PiperVoice

USE_CUDA = torch.cuda.is_available()
print("Piper use_cuda:", USE_CUDA)

tts_voice = PiperVoice.load(onnx_path, config_path=json_path, use_cuda=USE_CUDA)

def speak_final(text: str, out_wav="tts.wav", autoplay=True):
    print("TTS:", text)

    fh = None
    try:
        for chunk in tts_voice.synthesize(text):
            if fh is None:
                fh = wave.open(out_wav, "wb")
                fh.setframerate(chunk.sample_rate)
                fh.setsampwidth(chunk.sample_width)
                fh.setnchannels(chunk.sample_channels)
            fh.writeframes(chunk.audio_int16_bytes)
    finally:
        if fh is not None:
            fh.close()

    display(Audio(out_wav, autoplay=autoplay))
    return out_wav


## 5) Run MVP (Single-Turn)


In [ ]:
from rapidfuzz import fuzz


def lookup_numbers(raw_name: str):
    return CONTACTS.get(raw_name, {})


def match_option_from_reply(
    txt_norm: str,
    options,
    min_score: int = 70,
    allow_phone_type: bool = True,
):
    if allow_phone_type:
        inferred_type = extract_phone_type(txt_norm)
        if inferred_type in options:
            return inferred_type

    norm_options = [(opt, normalize_fa(opt)) for opt in options]
    for opt, opt_norm in norm_options:
        if opt_norm and opt_norm in txt_norm:
            return opt

    best_opt, best_score = None, 0
    for opt, opt_norm in norm_options:
        if not opt_norm:
            continue
        score = max(
            fuzz.token_set_ratio(txt_norm, opt_norm),
            fuzz.partial_ratio(txt_norm, opt_norm),
        )
        if score > best_score:
            best_opt, best_score = opt, score

    if best_opt and best_score >= min_score:
        return best_opt
    return None


def ask_and_get_phone_type(options):
    opts_text = ' یا '.join(options)
    speak_final(f'کدام شماره را می‌خواهید؟ {opts_text}')

    print(f'\nPROMPT: لطفاً یکی از گزینه‌ها را بگویید: {opts_text}')
    wav_path = record_to_wav('caller.wav')
    asr_text = transcribe_fa(wav_path)
    txt_norm = normalize_fa(asr_text)

    print('ASR2(raw):', asr_text)
    print('ASR2(norm):', txt_norm)

    selected_option = match_option_from_reply(txt_norm, options)
    print('SelectedOption:', selected_option)
    return selected_option


def ask_and_get_name(options):
    opts_text = ' یا '.join(options)
    speak_final(f'منظورتان کدام شخص است؟ {opts_text}')

    print(f'\nPROMPT: لطفاً نام دقیق را بگویید: {opts_text}')
    wav_path = record_to_wav('caller.wav')
    asr_text = transcribe_fa(wav_path)
    txt_norm = normalize_fa(asr_text)

    print('ASR_name(raw):', asr_text)
    print('ASR_name(norm):', txt_norm)

    selected_name = match_option_from_reply(
        txt_norm,
        options,
        min_score=65,
        allow_phone_type=False,
    )
    print('SelectedName:', selected_name)
    return selected_name


def record_option_text(record):
    role = (record.get('role') or '').strip()
    if role:
        return f"{record['name']} {role}"
    return record['name']


def ask_and_get_contact_record(records):
    options = []
    option_to_record = {}
    for rec in records:
        option = record_option_text(rec)
        if option in option_to_record:
            continue
        options.append(option)
        option_to_record[option] = rec

    opts_text = ' یا '.join(options)
    speak_final(f'چند مورد پیدا شد. منظورتان کدام است؟ {opts_text}')

    print(f'\nPROMPT: لطفاً مورد دقیق را بگویید: {opts_text}')
    wav_path = record_to_wav('caller.wav')
    asr_text = transcribe_fa(wav_path)
    txt_norm = normalize_fa(asr_text)

    print('ASR_pick(raw):', asr_text)
    print('ASR_pick(norm):', txt_norm)

    selected_option = match_option_from_reply(
        txt_norm,
        options,
        min_score=62,
        allow_phone_type=False,
    )
    print('SelectedRecordOption:', selected_option)
    return option_to_record.get(selected_option)


def run_once_and_say_number():
    print('\nPROMPT: لطفاً درخواستتان را بگویید (مثلاً: شماره حسن زاده را می‌خواهم)')
    wav_path = record_to_wav('caller.wav')

    asr_text = transcribe_fa(wav_path)
    parsed = parse_request(asr_text)

    raw_name = parsed['name_raw']
    ptype = parsed['phone_type']
    candidate_names = parsed.get('candidate_names', [])
    candidate_records = parsed.get('candidate_records', [])
    preferred_label = parsed.get('preferred_label')
    role_dominant_intent = parsed.get('role_dominant_intent', False)

    chosen_record = None
    should_resolve_record_first = bool(candidate_records) and (
        role_dominant_intent or not raw_name
    )

    if should_resolve_record_first:
        if len(candidate_records) == 1:
            chosen_record = candidate_records[0]
        else:
            chosen_record = ask_and_get_contact_record(candidate_records)
            if chosen_record is None:
                speak_final('مورد دقیق را متوجه نشدم. لطفاً دوباره تماس بگیرید و کامل‌تر بگویید.')
                return

        raw_name = chosen_record['name']
        if ptype is None:
            ptype = chosen_record['label']

    if not raw_name and candidate_names:
        if len(candidate_names) == 1:
            raw_name = candidate_names[0]
        else:
            raw_name = ask_and_get_name(candidate_names)
            if raw_name is None:
                speak_final('نام دقیق را متوجه نشدم. لطفاً دوباره تماس بگیرید و نام کامل را بگویید.')
                return

    if not raw_name:
        speak_final('اسم را متوجه نشدم. لطفاً دوباره واضح‌تر بگویید.')
        return

    numbers = lookup_numbers(raw_name)
    if not numbers:
        speak_final(f'برای {raw_name} شماره‌ای پیدا نکردم.')
        return

    if ptype is None and preferred_label in numbers:
        ptype = preferred_label

    if ptype is None and len(numbers) > 1:
        options = list(numbers.keys())
        ptype = ask_and_get_phone_type(options)
        if ptype is None:
            speak_final('متوجه گزینه موردنظر نشدم. لطفاً دوباره تماس بگیرید و نام گزینه را واضح بگویید.')
            return

    if ptype is None:
        ptype = list(numbers.keys())[0]

    if ptype not in numbers:
        ptype = list(numbers.keys())[0]

    number = numbers[ptype]
    speak_final(f'‌شُمارِیِه {ptype} ، {raw_name} ، {number} است.')


run_once_and_say_number()


## 6) Gradio Demo UI (Share Without Exposing Notebook)
Run this cell to launch a public Gradio link for testing via microphone or audio upload.


In [ ]:
import os
import subprocess
import sys
import tempfile
import wave

# Install Gradio if missing in the runtime.
try:
    import gradio as gr
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "gradio"])
    import gradio as gr


def to_wav16k_mono(src_path: str) -> str:
    """Convert input audio to 16kHz mono WAV for ASR."""
    dst = tempfile.NamedTemporaryFile(prefix="ui_in_", suffix=".wav", delete=False).name
    subprocess.run(
        ["ffmpeg", "-y", "-i", src_path, "-ac", "1", "-ar", "16000", dst],
        check=True,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    return dst


def synthesize_tts_to_file(text: str) -> str:
    """Synthesize Persian TTS without notebook display side-effects."""
    out_wav = tempfile.NamedTemporaryFile(prefix="ui_tts_", suffix=".wav", delete=False).name
    fh = None
    try:
        for chunk in tts_voice.synthesize(text):
            if fh is None:
                fh = wave.open(out_wav, "wb")
                fh.setframerate(chunk.sample_rate)
                fh.setsampwidth(chunk.sample_width)
                fh.setnchannels(chunk.sample_channels)
            fh.writeframes(chunk.audio_int16_bytes)
    finally:
        if fh is not None:
            fh.close()
    return out_wav


def resolve_single_turn(asr_text: str):
    """Resolve one request without additional follow-up recording."""
    parsed = parse_request(asr_text)

    raw_name = parsed["name_raw"]
    ptype = parsed["phone_type"]
    candidate_names = parsed.get("candidate_names", [])
    candidate_records = parsed.get("candidate_records", [])
    preferred_label = parsed.get("preferred_label")
    role_dominant_intent = parsed.get("role_dominant_intent", False)

    # Prefer top record when role intent dominates or name is unresolved.
    if candidate_records and (role_dominant_intent or not raw_name):
        if len(candidate_records) == 1:
            chosen = candidate_records[0]
            raw_name = chosen["name"]
            if ptype is None:
                ptype = chosen["label"]
        else:
            options = [f"{r['name']} ({r['role']})" for r in candidate_records]
            msg = "چند مورد نزدیک پیدا شد. لطفاً دقیق‌تر بگویید: " + " یا ".join(options)
            return {"ok": False, "message": msg, "note": "Ambiguous contact records"}

    if not raw_name and candidate_names:
        if len(candidate_names) == 1:
            raw_name = candidate_names[0]
        else:
            msg = "چند نام نزدیک پیدا شد. لطفاً نام کامل را بگویید: " + " یا ".join(candidate_names)
            return {"ok": False, "message": msg, "note": "Ambiguous names"}

    if not raw_name:
        return {
            "ok": False,
            "message": "اسم را متوجه نشدم. لطفاً دوباره واضح‌تر بگویید.",
            "note": "No name resolved",
        }

    numbers = lookup_numbers(raw_name)
    if not numbers:
        return {
            "ok": False,
            "message": f"برای {raw_name} شماره‌ای پیدا نکردم.",
            "note": "Name resolved but no numbers",
        }

    if ptype is None and preferred_label in numbers:
        ptype = preferred_label
    if ptype is None:
        ptype = list(numbers.keys())[0]
    if ptype not in numbers:
        ptype = list(numbers.keys())[0]

    number = numbers[ptype]
    response = f"شُمارِه {ptype} ، {raw_name} ، {number} است."
    return {"ok": True, "message": response, "note": "Resolved"}


def gradio_infer(audio_path: str):
    """End-to-end UI handler: audio -> ASR -> retrieval -> TTS."""
    if not audio_path:
        return "", "", "No audio provided.", None

    try:
        wav_path = to_wav16k_mono(audio_path)
        asr_text = transcribe_fa(wav_path)
        result = resolve_single_turn(asr_text)
        tts_path = synthesize_tts_to_file(result["message"])
        return asr_text, result["message"], result["note"], tts_path
    except Exception as exc:
        return "", "", f"Runtime error: {exc}", None


with gr.Blocks() as demo:
    gr.Markdown("# Persian Voice Phone Directory - Demo")
    gr.Markdown("Speak Persian query (name or role). The app returns text + spoken answer.")

    audio_in = gr.Audio(
        sources=["microphone", "upload"],
        type="filepath",
        label="Input Audio",
    )
    run_btn = gr.Button("Run")

    asr_out = gr.Textbox(label="ASR Text")
    reply_out = gr.Textbox(label="Assistant Response")
    note_out = gr.Textbox(label="Status")
    tts_out = gr.Audio(type="filepath", label="TTS Output")

    run_btn.click(
        fn=gradio_infer,
        inputs=[audio_in],
        outputs=[asr_out, reply_out, note_out, tts_out],
    )

# share=True creates a public temporary link for testers.
demo.launch(share=True)
